<a href="https://colab.research.google.com/github/britoadriana/ScreenQA-PT-Eval/blob/main/Notebooks/GIT_EXAMPLE_Screen_complexity_classification_comented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install datasets opencv-python numpy matplotlib

In [ ]:
from datasets import load_dataset
dataset = load_dataset(
    "britoadriana/screen_qa_portuguese",
    split="test",
    cache_dir="/content/drive/MyDrive/screen_qa_portuguese"
)

print(dataset)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import io

In [ ]:
def analyze_iris_dataset(dataset_split):
    results = []

    print(f"Starting analysis of {len(dataset_split)} images...")

    for i, row in enumerate(dataset_split):
        # 1. Get the image and convert to OpenCV format
        pil_img = row['image'].convert('RGB')
        numpy_image = np.array(pil_img)

        # Convert RGB to BGR (OpenCV standard) and then to Grayscale
        img_bgr = numpy_image[:, :, ::-1].copy()
        grayscale = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # 2. IRIS ISC Pipeline (Enhancement and Edge Detection)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(grayscale)
        blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
        edges = cv2.Canny(blurred, 50, 150)

        # 3. Dilation (Density Preservation)
        kernel = np.ones((5, 5), np.uint8)
        dilated = cv2.dilate(edges, kernel, iterations=2)

        # 4. Metrics Calculation
        height, width = dilated.shape
        global_density = (np.count_nonzero(dilated) / (height * width)) * 100

        # Fragmentation (Contours)
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        fragmentation = len(contours)

        # 5. Store Results with ID (file_name or screen_id)
        results.append({
            'image_id': row.get('file_name', row.get('screen_id', f"img_{i}")),
            'global_density': round(global_density, 2),
            'fragmentation': fragmentation,
            'resolution': f"{width}x{height}",
            'associated_question': row.get('question', '') # Maintaining dataset context
        })

        if (i + 1) % 500 == 0:
            print(f"{i + 1} images processed...")

    return pd.DataFrame(results)

# Execute the analysis
analysis_results_df = analyze_iris_dataset(dataset) # 'dataset' refers to your Hugging Face Dataset object

# View the first rows of the table
analysis_results_df.head()